In [1]:
!pip install -q pandas requests openpyxl

import pandas as pd
import requests
import json
import time
from datetime import datetime

print("✅ Libraries installed")
print(f"📅 Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

✅ Libraries installed
📅 Started: 2026-01-30 18:48:50



In [2]:
def fetch_complete_collection(edition_id, collection_name, collection_name_ar):
    """
    Fetch complete hadith collection with ALL metadata
    """

    print(f"\n{'='*80}")
    print(f"📚 Fetching: {collection_name} ({collection_name_ar})")
    print(f"{'='*80}\n")

    url = f"https://cdn.jsdelivr.net/gh/fawazahmed0/hadith-api@1/editions/{edition_id}.json"

    try:
        print("🌐 Downloading from API...")
        start_time = time.time()

        response = requests.get(url, timeout=180)

        if response.status_code == 200:
            data = response.json()
            hadiths_raw = data.get('hadiths', [])
            metadata = data.get('metadata', {})

            elapsed = time.time() - start_time
            print(f"✅ Downloaded {len(hadiths_raw):,} hadiths in {elapsed:.1f}s")

            hadiths = []

            for idx, h in enumerate(hadiths_raw, 1):

                # Extract reference info
                ref = h.get('reference', {})
                if not isinstance(ref, dict):
                    ref = {}

                hadith_entry = {
                    # === CORE IDENTIFICATION ===
                    'hadith_id': f"{edition_id}_{h.get('hadithnumber', idx)}",
                    'collection': collection_name,
                    'collection_ar': collection_name_ar,
                    'hadith_number': h.get('hadithnumber', idx),

                    # === BOOK INFORMATION ===
                    'book_number': ref.get('book', ''),
                    'book_name_en': ref.get('book', 'Unknown Book'),
                    'book_name_ar': '',  # Will be populated if available in future

                    # === HADITH TEXT ===
                    'hadith_text_en': h.get('text', '').strip(),
                    'hadith_text_ar': h.get('arab', '').strip(),

                    # === CHAPTER/REFERENCE ===
                    'chapter': ref.get('hadith', ''),
                    'reference_full': f"{collection_name} {h.get('hadithnumber', idx)}",
                    'reference_short': f"{'Bukhari' if 'bukhari' in edition_id else 'Muslim'} {h.get('hadithnumber', idx)}",

                    # === METADATA ===
                    'narrator': '',  # Can be extracted from text if needed
                    'grade': 'Sahih',  # All Bukhari/Muslim are Sahih
                    'is_authenticated': True,
                    'authentication_note': f"Recorded in {collection_name}, universally accepted by Sunni scholars.",

                    # === TECHNICAL ===
                    'text_length': len(h.get('text', '')),
                    'has_arabic': bool(h.get('arab', '').strip()),
                    'language': 'en',
                }

                # Quality check
                if hadith_entry['hadith_text_en'] and len(hadith_entry['hadith_text_en']) > 30:
                    hadiths.append(hadith_entry)

                # Progress
                if idx % 1000 == 0:
                    print(f"  ⏳ Processed {idx:,}/{len(hadiths_raw):,} hadiths...")

            print(f"\n✅ Valid hadiths extracted: {len(hadiths):,}")
            return pd.DataFrame(hadiths)

        else:
            print(f"❌ HTTP Error: {response.status_code}")
            return pd.DataFrame()

    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()


In [3]:
print("\n" + "="*80)
print("🚀 HADITH DATABASE BUILDER - BUKHARI + MUSLIM ONLY")
print("="*80 + "\n")

collections = [
    {
        'id': 'eng-bukhari',
        'name': 'Sahih al-Bukhari',
        'name_ar': 'صحيح البخاري'
    },
    {
        'id': 'eng-muslim',
        'name': 'Sahih Muslim',
        'name_ar': 'صحيح مسلم'
    }
]

all_dataframes = []

for col in collections:
    df = fetch_complete_collection(col['id'], col['name'], col['name_ar'])

    if not df.empty:
        all_dataframes.append(df)
        print(f"\n✅ Added {len(df):,} hadiths from {col['name']}")

    time.sleep(3)  # Be respectful to API


🚀 HADITH DATABASE BUILDER - BUKHARI + MUSLIM ONLY


📚 Fetching: Sahih al-Bukhari (صحيح البخاري)

🌐 Downloading from API...
✅ Downloaded 7,589 hadiths in 0.2s
  ⏳ Processed 1,000/7,589 hadiths...
  ⏳ Processed 2,000/7,589 hadiths...
  ⏳ Processed 3,000/7,589 hadiths...
  ⏳ Processed 4,000/7,589 hadiths...
  ⏳ Processed 5,000/7,589 hadiths...
  ⏳ Processed 6,000/7,589 hadiths...
  ⏳ Processed 7,000/7,589 hadiths...

✅ Valid hadiths extracted: 7,571

✅ Added 7,571 hadiths from Sahih al-Bukhari

📚 Fetching: Sahih Muslim (صحيح مسلم)

🌐 Downloading from API...
✅ Downloaded 7,563 hadiths in 0.3s
  ⏳ Processed 1,000/7,563 hadiths...
  ⏳ Processed 2,000/7,563 hadiths...
  ⏳ Processed 3,000/7,563 hadiths...
  ⏳ Processed 4,000/7,563 hadiths...
  ⏳ Processed 5,000/7,563 hadiths...
  ⏳ Processed 6,000/7,563 hadiths...
  ⏳ Processed 7,000/7,563 hadiths...

✅ Valid hadiths extracted: 7,359

✅ Added 7,359 hadiths from Sahih Muslim


In [4]:
print("\n" + "="*80)
print("🔧 COMBINING AND CLEANING DATA")
print("="*80 + "\n")

if all_dataframes:
    # Combine
    combined_df = pd.concat(all_dataframes, ignore_index=True)
    print(f"📊 Total before cleaning: {len(combined_df):,}")

    # Remove duplicates (some hadiths may appear in both)
    before = len(combined_df)
    combined_df = combined_df.drop_duplicates(subset=['hadith_text_en'], keep='first')
    print(f"✓ Removed {before - len(combined_df):,} duplicates")

    # Remove very short hadiths
    before = len(combined_df)
    combined_df = combined_df[combined_df['text_length'] > 50]
    print(f"✓ Removed {before - len(combined_df):,} short hadiths (<50 chars)")

    # Remove NaN
    before = len(combined_df)
    combined_df = combined_df.dropna(subset=['hadith_text_en', 'reference_full'])
    print(f"✓ Removed {before - len(combined_df):,} with missing data")

    # Sort
    combined_df = combined_df.sort_values(
        ['collection', 'hadith_number']
    ).reset_index(drop=True)
    print(f"✓ Sorted by collection and number")

    # Save CSV
    filename_csv = 'hadith_bukhari_muslim_complete.csv'
    combined_df.to_csv(filename_csv, index=False, encoding='utf-8-sig')

    # Also save Excel for easier viewing
    filename_xlsx = 'hadith_bukhari_muslim_complete.xlsx'
    combined_df.to_excel(filename_xlsx, index=False, engine='openpyxl')

    print(f"\n{'='*80}")
    print("✅ DATASET SAVED")
    print(f"{'='*80}")
    print(f"📁 CSV: {filename_csv}")
    print(f"📁 Excel: {filename_xlsx}")
    print(f"📊 Total hadiths: {len(combined_df):,}")

    print("\n📋 Collection Breakdown:")
    print(combined_df['collection'].value_counts().to_string())

    print(f"\n📊 Statistics:")
    print(f"  With Arabic text: {combined_df['has_arabic'].sum():,} ({combined_df['has_arabic'].sum()/len(combined_df)*100:.1f}%)")
    print(f"  Average length: {combined_df['text_length'].mean():.0f} characters")
    print(f"  Min length: {combined_df['text_length'].min()}")
    print(f"  Max length: {combined_df['text_length'].max()}")

    # Sample
    print(f"\n{'='*80}")
    print("📖 SAMPLE HADITHS")
    print(f"{'='*80}\n")

    for i in range(3):
        h = combined_df.iloc[i]
        print(f"📚 {h['reference_short']}")
        print(f"   Book: {h['book_name_en']}")
        print(f"   Text: {h['hadith_text_en'][:200]}...")
        if h['has_arabic']:
            print(f"   Arabic: {h['hadith_text_ar'][:100]}...")
        print()

    # Download files
    print(f"\n{'='*80}")
    print("📥 PREPARING DOWNLOADS")
    print(f"{'='*80}\n")

    from google.colab import files

    print("Downloading CSV...")
    files.download(filename_csv)

    print("Downloading Excel...")
    files.download(filename_xlsx)

    print("\n✅ COMPLETE!")
    print(f"You now have {len(combined_df):,} authentic hadiths ready to use!")

else:
    print("\n❌ No data collected. Check your internet connection.")


🔧 COMBINING AND CLEANING DATA

📊 Total before cleaning: 14,930
✓ Removed 542 duplicates
✓ Removed 16 short hadiths (<50 chars)
✓ Removed 0 with missing data
✓ Sorted by collection and number

✅ DATASET SAVED
📁 CSV: hadith_bukhari_muslim_complete.csv
📁 Excel: hadith_bukhari_muslim_complete.xlsx
📊 Total hadiths: 14,372

📋 Collection Breakdown:
collection
Sahih al-Bukhari    7250
Sahih Muslim        7122

📊 Statistics:
  With Arabic text: 0 (0.0%)
  Average length: 473 characters
  Min length: 51
  Max length: 17032

📖 SAMPLE HADITHS

📚 Muslim 3
   Book: 0
   Text: Zuhayr bin Harb narrated to me, Ismā’īl, rather, Ibn Ulayyah narrated to us, on authority of Abd il-Azīz ibn Suhayb, on authority of Anas bin Mālik, that he said:‘Indeed what prevents me from relating...

📚 Muslim 4
   Book: 0
   Text: Muhammad bin Ubayd il-Ghubarī narrated to us, Abū Awānah narrated to us, on authority of Abī Hasīn, on authority of Abī Sālih, on authority of Abū Hurayrah, he said, the Messenger of Allah, peac

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ COMPLETE!
You now have 14,372 authentic hadiths ready to use!


In [5]:
# Create metadata JSON for the app
metadata = {
    'total_hadiths': len(combined_df),
    'collections': {
        'Sahih al-Bukhari': len(combined_df[combined_df['collection'] == 'Sahih al-Bukhari']),
        'Sahih Muslim': len(combined_df[combined_df['collection'] == 'Sahih Muslim'])
    },
    'has_arabic': int(combined_df['has_arabic'].sum()),
    'date_created': datetime.now().strftime('%Y-%m-%d'),
    'columns': list(combined_df.columns),
    'sample_references': [
        combined_df.iloc[0]['reference_short'],
        combined_df.iloc[100]['reference_short'],
        combined_df.iloc[1000]['reference_short']
    ]
}

with open('dataset_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("\n📋 Metadata saved to: dataset_metadata.json")

files.download('dataset_metadata.json')


📋 Metadata saved to: dataset_metadata.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>